In [ ]:
import io
import os
import zipfile
import requests
import pandas as pd
import seaborn as sns
import geopandas as gpd
import contextily as ctx
from matplotlib.colors import LogNorm

# =============================================
# 1. CẤU HÌNH ĐƯỜNG DẪN ĐỒNG BỘ VỀ FOLDER GỐC
# =============================================
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(BASE_DIR, '../../')) 

# Chỉ định đích danh các folder nằm trực tiếp dưới thư mục lớn nhất
RAW_DIR = os.path.join(PROJECT_ROOT, 'raw')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'processed')
FIGURES_DIR = os.path.join(PROJECT_ROOT, 'figures') # Lưu thẳng vào Data/figures

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

DATA_FILE = os.path.join(PROCESSED_DIR, 'yellow_tripdata_2021_full.parquet')
SHAPEFILE_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip"

# 🔥 ĐỊNH VỊ CHÍNH XÁC: Chỉ định đường dẫn tới thư mục 'archive' theo đúng ảnh thực tế
SHAPEFILE_PATH_ARCHIVE = os.path.join(RAW_DIR, 'archive', 'taxi_zones.shp')
SHAPEFILE_PATH_DIRECT = os.path.join(RAW_DIR, 'taxi_zones.shp')

# =============================================
# 2. KIỂM TRA VÀ TẢI DỮ LIỆU BẢN ĐỒ
# =============================================
print("🗺️ Đang kiểm tra cấu trúc dữ liệu bản đồ...")

# Quyết định chọn đường dẫn chính xác dựa trên sự tồn tại thực tế của file
if os.path.exists(SHAPEFILE_PATH_ARCHIVE):
    ACTUAL_SHAPEFILE_PATH = SHAPEFILE_PATH_ARCHIVE
elif os.path.exists(SHAPEFILE_PATH_DIRECT):
    ACTUAL_SHAPEFILE_PATH = SHAPEFILE_PATH_DIRECT
else:
    # Nếu chưa tồn tại file ở cả 2 nơi, tiến hành tải mới từ server AWS
    print(f"⚠️ Không thấy file shapefile. Đang tự động tải từ TLC NYC...")
    try:
        r = requests.get(SHAPEFILE_URL)
        z = zipfile.ZipFile(io.BytesIO(r.content))
        z.extractall(RAW_DIR)
        print("✅ Đã tải và giải nén bản đồ taxi thành công.")
        
        # Kiểm tra lại sau khi giải nén mới xem file nằm ở đâu
        if os.path.exists(SHAPEFILE_PATH_ARCHIVE):
            ACTUAL_SHAPEFILE_PATH = SHAPEFILE_PATH_ARCHIVE
        else:
            ACTUAL_SHAPEFILE_PATH = SHAPEFILE_PATH_DIRECT
    except Exception as e:
        print(f"❌ Lỗi tải hoặc giải nén Shapefile: {e}")
        exit()

print(f"🎯 ĐÃ ĐỊNH VỊ ĐƯỢC FILE BẢN ĐỒ TẠI:\n 👉 {ACTUAL_SHAPEFILE_PATH}")

# Đọc file bản đồ bằng Geopandas từ đường dẫn chuẩn xác vừa tìm thấy
try:
    nyc_map = gpd.read_file(ACTUAL_SHAPEFILE_PATH)
    nyc_map = nyc_map.to_crs(epsg=3857)
    print("✅ Đã nạp bản đồ không gian địa lý thành công vào bộ nhớ!")
except Exception as e:
    print(f"❌ Lỗi đọc Shapefile bằng Geopandas: {e}")
    exit()

# CẤU HÌNH STYLE NỀN TRẮNG CHUẨN CHUYÊN NGHIỆP
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 12,
    'text.color': '#333333',
    'axes.labelcolor': '#333333',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titleweight': 'bold',
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.unicode_minus': False
})

# =============================================
# 3. ĐỌC DỮ LIỆU CHUYẾN ĐI PARQUET
# =============================================
print("\n📥 Đang kiểm tra dữ liệu chuyến đi...")
if not os.path.exists(DATA_FILE):
    print(f"❌ LỖI: Không tìm thấy file dữ liệu Parquet lớn tại: {DATA_FILE}")
    exit()
else:
    try:
        cols_to_load = ['tpep_pickup_datetime', 'PULocationID', 'trip_distance']
        df = pd.read_parquet(DATA_FILE, columns=cols_to_load)
        print(f"✅ Đã nạp xong {len(df):,} chuyến đi thực tế từ file Parquet.")
    except Exception as e:
        print(f"❌ Lỗi đọc file Parquet: {e}")
        exit()

# =============================================
# BIỂU ĐỒ 1: BẢN ĐỒ MẬT ĐỘ ĐÓN KHÁCH (LOG SCALE)
# =============================================
print("\n--- 1. Vẽ Bản đồ mật độ (Log Scale) ---")
pickup_counts = df['PULocationID'].value_counts().reset_index()
pickup_counts.columns = ['LocationID', 'trip_count']
nyc_map['LocationID'] = nyc_map['LocationID'].astype(int)
pickup_counts['LocationID'] = pickup_counts['LocationID'].astype(int)

map_data = nyc_map.merge(pickup_counts, on='LocationID', how='left')
map_data['trip_count'] = map_data['trip_count'].fillna(1)

fig, ax = plt.subplots(figsize=(12, 12))
map_data.plot(
    column='trip_count',
    cmap='Spectral_r',
    norm=LogNorm(vmin=map_data['trip_count'].min(), vmax=map_data['trip_count'].max()),
    linewidth=0.5,
    edgecolor='0.6',
    legend=True,
    legend_kwds={'label': "Số lượng chuyến đón khách (Log Scale)", 'shrink': 0.6},
    ax=ax,
    alpha=0.85
)

try:
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
except Exception as e:
    print(f"⚠️ Không load được hình nền bản đồ đô thị: {e}")

ax.set_xlim(-8260000, -8200000) 
ax.set_ylim(4940000, 5000000)
ax.set_axis_off()
ax.set_title('Bản đồ Phân bổ Mật độ Đón khách Taxi năm 2021', fontsize=16, fontweight='bold', pad=10)
plt.tight_layout()

output_path_1 = os.path.join(FIGURES_DIR, 'map_pickup_density_2021_log.png')
plt.savefig(output_path_1, dpi=300, bbox_inches='tight')
print(f"✅ Đã lưu bản đồ mật độ tại: {output_path_1}")
plt.show()
plt.close()

# =============================================
# BIỂU ĐỒ 2: XU HƯỚNG THEO GIỜ (LINE CHART)
# =============================================
print("\n--- 2. Vẽ Biểu đồ xu hướng theo giờ ---")
if not pd.api.types.is_datetime64_any_dtype(df['tpep_pickup_datetime']):
    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])

df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
hourly_counts = df.groupby('pickup_hour').size().reset_index(name='trip_count')

plt.figure(figsize=(12, 6))
sns.lineplot(data=hourly_counts, x='pickup_hour', y='trip_count', color='#FF5733', linewidth=2.5, marker='o')
plt.title('Tổng số chuyến đi phát sinh theo Giờ trong ngày (Nhu cầu mạng lưới)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Khung giờ trong ngày (0h - 23h)', fontsize=12)
plt.ylabel('Tổng lượng chuyến xe', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(range(0, 24))

output_path_2 = os.path.join(FIGURES_DIR, 'chart_trips_by_hour.png')
plt.savefig(output_path_2, dpi=300, bbox_inches='tight')
print(f"✅ Đã lưu đồ thị giờ tại: {output_path_2}")
plt.show()
plt.close()

# =============================================
# BIỂU ĐỒ 3: PHÂN BỐ THEO THỨ (BAR CHART)
# =============================================
print("\n--- 3. Vẽ Biểu đồ phân bổ theo thứ ---")
df['pickup_weekday'] = df['tpep_pickup_datetime'].dt.day_name()
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['pickup_weekday'] = pd.Categorical(df['pickup_weekday'], categories=days_order, ordered=True)

weekday_counts = df.groupby('pickup_weekday', observed=False).size().reset_index(name='trip_count')

plt.figure(figsize=(10, 6))
sns.barplot(data=weekday_counts, x='pickup_weekday', y='trip_count', order=days_order, hue='pickup_weekday', legend=False, palette='viridis', edgecolor='black', alpha=0.8)
plt.title('Phân bố tổng lượng chuyến đi theo Thứ trong tuần', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Các ngày trong tuần', fontsize=12)
plt.ylabel('Tổng số chuyến xe', fontsize=12)

output_path_3 = os.path.join(FIGURES_DIR, 'chart_trips_by_weekday.png')
plt.savefig(output_path_3, dpi=300, bbox_inches='tight')
print(f"✅ Đã lưu đồ thị thứ tại: {output_path_3}")
plt.show()
plt.close()

# =============================================
# BIỂU ĐỒ 4: PHÂN BỐ QUÃNG ĐƯỜNG (HISTOGRAM)
# =============================================
print("\n--- 4. Vẽ Histogram phân bổ quãng đường ---")
filtered_distance = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 20)]

plt.figure(figsize=(12, 6))
sns.histplot(filtered_distance['trip_distance'], bins=50, kde=True, color='teal', edgecolor='black', alpha=0.7)
plt.title('Biểu đồ tần suất Phân bố Quãng đường chuyến đi (< 20 dặm)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Quãng đường di chuyển thực tế (Đơn vị: Dặm / Miles)", fontsize=12)
plt.ylabel("Tần suất lặp lại (Số lượng chuyến)", fontsize=12)

output_path_4 = os.path.join(FIGURES_DIR, 'chart_distance_distribution.png')
plt.savefig(output_path_4, dpi=300, bbox_inches='tight')
print(f"✅ Đã lưu đồ thị quãng đường tại: {output_path_4}")
plt.show()
plt.close()

print(f"\n🎉 QUÁ TUYỆT VỜI! Toàn bộ 4 ảnh phân tích chất lượng cao đã chui thẳng vào folder figures gốc:\n 👉 {os.path.abspath(FIGURES_DIR)}")